# Entrega 3: Preprocesamiento Estructural, Modelo y Métricas

**Proyecto:** Dinámica del comercio mundial: exportaciones e importaciones por país y región geográfica (1989-2023)

**Contexto:** En el Entregable 2, el dataset fue perfilado y limpiado exhaustivamente (tratamiento de nulos, tipado de datos y homogeneización de entidades). Por lo tanto, el objetivo de este notebook no es repetir la limpieza de calidad de datos, sino ejecutar el **preprocesamiento estructural necesario para llegar al modelo analítico final** que consumirá Tableau.

In [7]:
import pandas as pd
import numpy as np
import time
import os

import warnings
warnings.filterwarnings('ignore')

## 1. Carga de Datos Limpios del Entregable 2

In [8]:
try:
    df_base = pd.read_csv('../data/processed/dataset_limpio_entrega2_consolidado.csv')
    print(f'Dataset limpio cargado exitosamente: {df_base.shape[0]} filas y {df_base.shape[1]} columnas.')
except FileNotFoundError:
    print('Advertencia: Ejecuta primero el notebook del Entregable 2. Generando dummy temporal.')
    df_base = pd.DataFrame({
        'Year': np.random.randint(1989, 2024, 10000),
        'Partner Name': np.random.choice([f'Country_{i}' for i in range(250)], 10000),
        'Region': np.random.choice([f'Region_{i}' for i in range(6)], 10000),
        'World Growth (%)': np.random.normal(3, 1, 10000),
        'Export (US$ Million)': np.random.uniform(0, 5000, 10000),
        'AHS Weighted Average (%)': np.random.uniform(0, 20, 10000)
    })

Dataset limpio cargado exitosamente: 7783 filas y 38 columnas.


## 2. Preprocesamiento Estructural para Llegar al Modelo
Dado que los datos ya están limpios, los pasos requeridos para transformar esta 'Tabla Plana' en un Modelo Relacional (Esquema en Estrella) son de naturaleza arquitectónica:

In [9]:
# PASO 1: Validación de Granularidad y Unicidad (Para evitar explosión de Joins en el modelo)
duplicados = df_base.duplicated(subset=['Partner Name', 'Year']).sum()
print(f"1. Verificando cardinalidad País-Año: {duplicados} duplicados encontrados.")
if duplicados > 0:
    df_base = df_base.drop_duplicates(subset=['Partner Name', 'Year'])

# PASO 2: Normalización (Resolución de Dependencias Transitivas para el Modelo)
# Justificación: Para modelar en Estrella, necesitamos separar atributos que no dependen directamente del cruce transaccional.
dim_country_cols = ['Partner Name']
if 'Region' in df_base.columns: dim_country_cols.append('Region')
    
dim_time_cols = ['Year']
if 'World Growth (%)' in df_base.columns: dim_time_cols.append('World Growth (%)')

dim_country = df_base[dim_country_cols].drop_duplicates().reset_index(drop=True)
dim_time = df_base[dim_time_cols].drop_duplicates().reset_index(drop=True)
print("2. Matrices dimensionales aisladas exitosamente para evitar redundancia de la Tabla Plana.")

# PASO 3: Generación de Surrogate Keys (Claves Subrogadas)
# Justificación: Los motores analíticos (como Tableau Hyper) procesan joins de números enteros mucho más rápido que cadenas de texto.
dim_country.insert(0, 'dim_country_sk', range(1, 1 + len(dim_country)))
dim_time.insert(0, 'dim_time_sk', range(1, 1 + len(dim_time)))
print("3. Surrogate Keys generadas para optimización del motor relacional.")

# PASO 4: Construcción de la Tabla de Hechos (Fact Table)
# Justificación: Reemplazamos los strings por las llaves generadas para crear la matriz central puramente numérica.
fact_trade = df_base.merge(dim_country[['Partner Name', 'dim_country_sk']], on='Partner Name', how='left')
fact_trade = fact_trade.merge(dim_time[['Year', 'dim_time_sk']], on='Year', how='left')

cols_to_drop = [c for c in df_base.columns if c in dim_country.columns or c in dim_time.columns]
fact_trade = fact_trade.drop(columns=cols_to_drop)
print("4. Tabla de Hechos central generada.")

1. Verificando cardinalidad País-Año: 0 duplicados encontrados.
2. Matrices dimensionales aisladas exitosamente para evitar redundancia de la Tabla Plana.
3. Surrogate Keys generadas para optimización del motor relacional.
4. Tabla de Hechos central generada.


## 3. Discusión y Evaluación de Opciones de Modelo
Comparamos la **Opción A (Tabla Plana original del Entregable 2)** vs la **Opción B (Esquema en Estrella generado en el preprocesamiento)** usando métricas computacionales exactas.

In [10]:
resultados_bench = {}

# Métrica 1: Eficiencia de Huella de Memoria RAM (KB)
mem_obt = df_base.memory_usage(deep=True).sum() / 1024
mem_star = (dim_country.memory_usage(deep=True).sum() + 
            dim_time.memory_usage(deep=True).sum() + 
            fact_trade.memory_usage(deep=True).sum()) / 1024

resultados_bench['Memoria RAM (KB)'] = {'Tabla Plana (OBT)': round(mem_obt, 2), 'Esquema Estrella': round(mem_star, 2)}

# Métrica 2: Riesgo de Agregación (Fan-out Trap sobre el Promedio Mundial)
# Demostramos cómo la Tabla Plana distorsiona métricas globales al replicarlas.
if 'World Growth (%)' in df_base.columns:
    avg_obt = df_base['World Growth (%)'].mean() # Incorrecto matemático
    avg_star = dim_time['World Growth (%)'].mean() # Promedio real por año único
    resultados_bench['Distorsión (Avg World Growth)'] = {'Tabla Plana (OBT)': f'{avg_obt:.3f}% (FALSO)', 'Esquema Estrella': f'{avg_star:.3f}% (REAL)'}

print("Métricas de evaluación calculadas.")

Métricas de evaluación calculadas.


## 4. Tabla Comparativa de Modelos y Exportación
Fundamentamos empíricamente la elección de la arquitectura.

In [11]:
df_comparativo = pd.DataFrame(resultados_bench).T
df_comparativo.index.name = 'Métrica Evaluada'
df_comparativo.reset_index(inplace=True)

# Justificación ligada a las métricas:
df_comparativo['Justificación de Decisión para el Proyecto'] = [
    "El modelo Estrella ahorra memoria descartando los textos duplicados; necesario para fluidez en Tableau.",
    "La Estrella resuelve el problema del Fan-Out. Evita promedios distorsionados sin forzar el uso de cálculos LoD complejos."
]

display(df_comparativo)

os.makedirs('../outputs', exist_ok=True)
df_comparativo.to_csv('../outputs/tabla_comparativa_modelos.csv', index=False)
print("\nTabla comparativa de justificación de modelo exportada a /outputs/")

,Métrica Evaluada,Tabla Plana (OBT),Esquema Estrella,Justificación de Decisión para el Proyecto
0,Memoria RAM (KB),3432.12,3000.59,El modelo Estrella ahorra memoria descartando ...
1,Distorsión (Avg World Growth),1.895% (FALSO),1.959% (REAL),La Estrella resuelve el problema del Fan-Out. ...



Tabla comparativa de justificación de modelo exportada a /outputs/


## 5. Exportación del Modelo Seleccionado
Se exporta el Esquema en Estrella para la capa lógica de Tableau.

In [12]:
os.makedirs('../outputs/tableau_sources', exist_ok=True)
fact_trade.to_csv('../outputs/tableau_sources/Fact_Trade.csv', index=False)
dim_country.to_csv('../outputs/tableau_sources/Dim_Country.csv', index=False)
dim_time.to_csv('../outputs/tableau_sources/Dim_Time.csv', index=False)

print("Pipeline finalizado. Fuentes listas para conectar.")

Pipeline finalizado. Fuentes listas para conectar.
